## Импорты библиотек

In [1]:
import math
import random
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import shap

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

## Конфигурация путей и фиксация seed

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DATA_DIR = Path('data')

DATA_DIR.mkdir(parents=True, exist_ok=True)

## Генерация датасета сотрудников

Создает датасет с реалистичным распределением уровней, зарплат, метрик вовлеченности и риска текучести

Формирование признаков имитирует корпоративные HR-данные

In [3]:
def generate_employees(n: int) -> pd.DataFrame:
    today = datetime.now()
    DEPARTMENTS = ['IT', 'Sales', 'Marketing', 'Finance', 'HR', 'Operations', 'Legal']                   # Отдел - для полноты датасета
    DEPT_WEIGHTS = [0.28, 0.18, 0.12, 0.15, 0.10, 0.12, 0.05]                                            # Распределение по отделам
    LEVELS = ['L1', 'L2', 'L3', 'L4', 'L5', 'L6']                                                        # Уровень
    LEVEL_WEIGHTS = [0.15, 0.25, 0.28, 0.18, 0.10, 0.04]                                                 # Распределение по уровням
    LOCATIONS = ['Москва', 'СПб', 'Удаленно', 'Казань', 'Екатеринбург']                                  # Локация
    LOC_MULT = {'Москва': 1.15, 'СПб': 1.05, 'Удаленно': 1.00, 'Казань': 0.85, 'Екатеринбург': 0.82}     # Коэффициенты зарплат по локации
    EMP_TYPES = ['Полная занятость', 'Частичная', 'Контракт']                                            # Тип занятости - для расчета эффективности
    LANGUAGES = ['Python', 'SQL', 'JavaScript', 'Java', 'Go', 'TypeScript', 'C#', 'Rust', 'Scala', 'R']  # Языки программирования
    TECHNOLOGIES = ['DevOps', 'Cloud', 'ML', 'Data Engineering', 'Security', 'Docker', 
                    'Kubernetes', 'Airflow', 'dbt', 'PostgreSQL', 'React', 'Node.js', 'Spark', 
                    'TensorFlow', 'FastAPI']                                                             # Технилогии
    BASE_SALARY = {'L1': 85000, 'L2': 135000, 'L3': 195000, 'L4': 280000, 'L5': 410000, 'L6': 600000}    # Базовая ставка по уровню
    
    records = []
    for i in range(1, n + 1):
        eid = f"EMP-{10000 + i}"
        dept = random.choices(DEPARTMENTS, weights=DEPT_WEIGHTS, k=1)[0]
        level = np.random.choice(LEVELS, p=LEVEL_WEIGHTS)
        loc = random.choice(LOCATIONS)
        emp_type = random.choices(EMP_TYPES, weights=[0.75, 0.15, 0.10], k=1)[0]
        fte = 1.0 if emp_type == 'Полная занятость' else random.choice([0.5, 0.75])
        
        tenure = min(15.0, np.random.exponential(scale=3.5))
        time_in_role = min(tenure, np.random.exponential(scale=2.0))
        time_since_promo = min(tenure, np.random.exponential(scale=1.5))
        
        perf = np.clip(np.random.beta(2.2, 1.4) * 4.2 + 0.8, 1.0, 5.0)
        engagement = np.clip(np.random.normal(3.9, 0.55), 1.0, 5.0)
        
        logit = (-1.5 + 
                 0.5 * (1.0 if tenure < 1.5 else 0.0) - 
                 0.25 * (perf / 5.0) + 
                 0.2 * (time_since_promo / 2.0) - 
                 0.15 * (engagement / 5.0) + 
                 np.random.normal(0, 0.25))
        turnover_risk = np.clip(1 / (1 + np.exp(-logit)), 0.05, 0.65)
        
        base = BASE_SALARY[level] * LOC_MULT[loc] * fte
        tenure_effect = 1 + 0.12 * min(tenure, 3) + 0.03 * max(0, tenure - 3)
        market_noise = np.random.normal(0, 0.08)
        salary = int(np.clip(base * tenure_effect * (1 + market_noise), 60000, 900000))
        
        n_langs = random.randint(1, 3)
        n_techs = random.randint(2, 4)
        emp_languages = random.sample(LANGUAGES, k=min(n_langs, len(LANGUAGES)))
        emp_technologies = random.sample(TECHNOLOGIES, k=min(n_techs, len(TECHNOLOGIES)))
        
        training_hrs = max(0, int(np.random.poisson(22) + np.random.normal(0, 5)))
        bonus_pct = random.choice([0.0, 0.10, 0.15, 0.20, 0.25])
        
        hire_date = today - timedelta(days=int(tenure * 365))
        last_promo = hire_date + timedelta(days=int(time_since_promo * 365)) if time_since_promo < tenure else hire_date
        manager_id = f"EMP-{10000 + random.randint(1, n)}" if level != 'L1' else None
        
        records.append({
            'employee_id': eid, 'department': dept, 'job_level': level, 'location': loc,
            'employment_type': emp_type, 'fte_percentage': fte, 'tenure_years': round(tenure, 1),
            'time_in_current_role_years': round(time_in_role, 1), 'performance_rating': round(perf, 2),
            'engagement_score': round(engagement, 2), 'base_salary_monthly': salary,
            'bonus_target_pct': bonus_pct, 'turnover_risk_score': round(turnover_risk, 3),
            'languages': emp_languages, 'technologies': emp_technologies,
            'training_hours_ytd': training_hrs, 'manager_id': manager_id,
            'hire_date': hire_date.strftime('%Y-%m-%d'),
            'last_promotion_date': last_promo.strftime('%Y-%m-%d')
        })
    return pd.DataFrame(records)

Создается 2 датасета из 200 сотрудников, один для обучения модели, другой для инференса

In [4]:
num_employees = 400

df_employees = generate_employees(n=num_employees)
print(f"Сгенерировано {len(df_employees)} данных о сотрудниках")

df_train, df_inference = train_test_split(df_employees, test_size=0.5, random_state=42)

df_train.to_csv(DATA_DIR / 'employees_train.csv', index=False, encoding='utf-8-sig')
print(f"Файл с сотрудниками для обучения модели сохранен в {DATA_DIR / 'employees_train.csv'}")
df_inference.to_csv(DATA_DIR / 'employees_inference.csv', index=False, encoding='utf-8-sig')
print(f"Файл с сотрудниками для инференса сохранен в {DATA_DIR / 'employees_inference.csv'}")

Сгенерировано 400 данных о сотрудниках
Файл с сотрудниками для обучения модели сохранен в data\employees_train.csv
Файл с сотрудниками для инференса сохранен в data\employees_inference.csv


Вывод первых 5 сотрудников на предпросмотр

In [5]:
df_employees.head()

,employee_id,department,job_level,location,employment_type,fte_percentage,tenure_years,time_in_current_role_years,performance_rating,engagement_score,base_salary_monthly,bonus_target_pct,turnover_risk_score,languages,technologies,training_hours_ytd,manager_id,hire_date,last_promotion_date
0,EMP-10001,Finance,L2,Москва,Полная занятость,1.00,10.5,2.6,3.53,4.77,236988,0.25,0.184,[JavaScript],"[Node.js, Cloud]",19,EMP-10045,2015-10-20,2017-03-02
1,EMP-10002,Finance,L2,Москва,Полная занятость,1.00,2.6,1.1,2.31,4.71,204858,0.25,0.147,[Scala],"[PostgreSQL, DevOps]",15,EMP-10102,2023-09-23,2024-03-29
2,EMP-10003,Finance,L3,Екатеринбург,Полная занятость,1.00,0.7,0.1,3.30,3.89,183804,0.20,0.185,"[Go, Python]","[Spark, FastAPI, ML, Node.js]",12,EMP-10175,2025-09-04,2025-09-04
3,EMP-10004,IT,L3,СПб,Контракт,0.75,0.7,0.7,3.78,3.84,147021,0.15,0.213,[C#],"[Cloud, Docker]",23,EMP-10310,2025-08-12,2025-08-12
4,EMP-10005,IT,L2,Москва,Полная занятость,1.00,2.7,0.3,2.36,4.47,192418,0.25,0.218,"[C#, SQL, Go]","[TensorFlow, React]",26,EMP-10186,2023-08-05,2026-01-08
